# Locked-test evaluation of the selected assessor (Qwen2.5-VL-7B)

Final step of the protocol: **train → select on validation → open the locked test once**.
Nothing is trained here. The frozen, validation-selected Qwen2.5-VL-7B checkpoint (epoch 0) is scored once.

The run first re-scores the 7,861 validation rows and must reproduce the recorded outcome MCC (0.678, within 0.005).
Only then are the 7,855 locked-test rows opened. If the check fails, the test stays closed.

**Resumable and runs in the background.** Rows are scored in chunks of 400 and each finished chunk is saved.
Cell 4 starts the run in the background, so closing VS Code or losing the connection does not stop it.
After a power cut or crash, run cells 1 to 4 again: finished chunks are reused, only the rest is scored.

Order: cells 1, 2, 3, 4 once. Then cell 5 whenever you want to see progress. Cell 6 when the log says `written`.


In [ ]:
# 1. Paths. Edit only this cell if your machine differs.
from pathlib import Path
import os, sys, json, subprocess

REPO_ROOT = Path('/home/aiub/kiyas/webagent').resolve()
DATA_HOME = Path('/home/aiub/kiyas/webagent_full/data').resolve()
WORKSPACE = Path('/home/aiub/kiyas/webagent_comparison').resolve()   # where the DGX runs wrote checkpoints
OUT_DIR = REPO_ROOT / 'results' / 'locked_test'
NUM_WORKERS = 8
os.environ['HF_HOME'] = str(WORKSPACE / 'hf_cache')
sys.path.insert(0, str(REPO_ROOT / 'src'))
os.chdir(REPO_ROOT)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
# 2. Data roots. This time split_test.json MUST be present.
from web_agent.data.recovery_supplement import resolve_supplement_root

roots = sorted({p.parent.resolve() for p in (DATA_HOME / 'original').rglob('split_test.json')
                if p.parent.joinpath('split_val.json').is_file() and p.parent.joinpath('images').is_dir()})
assert len(roots) == 1, f'expected one data root with split_test.json, found {roots}'
ORIGINAL_ROOT = roots[0]
SUPPLEMENT_ROOT = resolve_supplement_root(DATA_HOME / 'supplement')
print('original:', ORIGINAL_ROOT)
print('supplement:', SUPPLEMENT_ROOT)


In [ ]:
# 3. The validation-selected checkpoint and its run report.
#    The DGX workspace copy is used if present; otherwise the Git LFS copy in the repo.
MODEL_ID = 'qwen25vl_7b_gold_v2_8_dgx'
RUN = Path(MODEL_ID) / 'seed_42' / 'full'
CKPT_NAME = 'checkpoints/Y_QWEN25VL_7B_GOLD_V2_8_DGX_FULL_SEED42/best_e0_outcome-mcc0.678.ckpt'
BASES = [WORKSPACE / 'outputs' / 'model_comparison',
         REPO_ROOT / 'webagent_comparison' / 'outputs' / 'model_comparison']

def first_real(paths, min_size=1):
    for p in paths:
        if p.is_file() and p.stat().st_size >= min_size:
            return p
    raise FileNotFoundError(f'none found (run git lfs pull?): {paths}')

CKPT = first_real([b / RUN / CKPT_NAME for b in BASES], min_size=1_000_000)
REPORT = first_real([b / RUN / 'report.json' for b in BASES])
print('checkpoint:', CKPT)
print('report:', REPORT)


In [ ]:
# 4. Start the run in the background (validation check, then the locked test).
#    Safe to run again after a crash: finished chunks are reused.
OUT = OUT_DIR / f'{MODEL_ID}.json'
LOG = OUT_DIR / f'{MODEL_ID}.log'
cmd = [sys.executable, '-u', 'scripts/evaluate_locked_test.py',
       '--model-id', MODEL_ID, '--checkpoint', str(CKPT), '--report', str(REPORT),
       '--data-root', str(ORIGINAL_ROOT), '--supplement-root', str(SUPPLEMENT_ROOT),
       '--num-workers', str(NUM_WORKERS), '--out', str(OUT)]
running = subprocess.run(['pgrep', '-f', 'evaluate_locked_test.py'], capture_output=True, text=True).stdout.split()
if running:
    # Never start a second copy; the running one keeps going. Watch it with cell 5.
    print(f'Already running (pids {running}). Nothing started. Use cell 5 to watch progress.')
elif OUT.is_file():
    print('Already finished:', OUT, '- run cell 6.')
else:
    with open(LOG, 'a') as log:
        proc = subprocess.Popen(['nohup', *cmd], stdout=log, stderr=subprocess.STDOUT,
                                cwd=REPO_ROOT, start_new_session=True)
    print('started, pid', proc.pid)
print('log:', LOG)


In [ ]:
# 5. Progress. Run this cell any time; it only reads the log.
import subprocess
print(subprocess.run(['tail', '-n', '15', str(LOG)], capture_output=True, text=True).stdout)
alive = subprocess.run(['pgrep', '-f', 'evaluate_locked_test.py'], capture_output=True, text=True).stdout.split()
print('RUNNING' if alive else ('FINISHED' if OUT.is_file() else 'STOPPED: run cells 1 to 4 again to resume'))


In [ ]:
# 5b. LIVE progress bar: updates by itself every 10 s until the run finishes (stop the cell any time; the run continues).
import time, importlib.util
from IPython.display import clear_output
spec = importlib.util.spec_from_file_location('ltp', REPO_ROOT / 'scripts' / 'locked_test_progress.py')
ltp = importlib.util.module_from_spec(spec); spec.loader.exec_module(ltp)
try:
    while True:
        text, stop = ltp.snapshot(MODEL_ID)
        clear_output(wait=True); print(text)
        if stop:
            break
        time.sleep(10)
except KeyboardInterrupt:
    print('stopped watching (the evaluation keeps running)')


In [ ]:
# 6. Validation next to test (after the log says `written`).
import pandas as pd
r = json.loads(OUT.read_text())
rows = [{'split': 'validation', **r['validation_check']['metrics']},
        {'split': 'test', **r['test_metrics']}]
table = pd.DataFrame(rows).set_index('split').astype(float).round(3)
table.to_csv(OUT_DIR / 'qwen25vl_7b_validation_vs_test.csv')
print('test rows:', r['test_rows'], '| checkpoint sha256:', r['checkpoint_sha256'][:12])
table
